# M5 Forecasting - Model Training and Evaluation

This notebook demonstrates:
- Feature engineering
- Model training with XGBoost and LightGBM
- Baseline comparison (naive lag-1, seasonal lag-7, rolling mean 7d) vs ML models
- Model evaluation and comparison
- After training: metrics, learning curves, feature importance per model, daily actual vs predictions + baselines, then sample series and residual plot for each model (e.g. XGBoost and LightGBM)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

sys.path.append('..')

from src.pipeline.train_pipeline import TrainingPipeline
from src.utils.config import load_config
from src.evaluation.plots import display_training_report_plots

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Initialize training pipeline
pipeline = TrainingPipeline(config_path='../configs/config.m5_optimized.yaml')

In [ ]:
# Run training pipeline (charts are shown in the next cell)
models, results, plot_ctx = pipeline.run(return_plot_inputs=True)

In [ ]:
# Figures: metrics → learning curves + FI per model → daily (ML + baselines) →
# sample series + residual scatter for each trained model (xgboost & lightgbm by default).
from IPython.display import Markdown, display

display(Markdown("### Train / eval — figures"))
training_cfg = pipeline.config.get("training", {}) or {}
display_training_report_plots(
    test_results=results,
    test_df=plot_ctx["test_df"],
    feature_cols=plot_ctx["feature_cols"],
    model_trainer=pipeline.model_trainer,
    date_col=training_cfg.get("date_col", "date"),
    target_col=training_cfg.get("target_col", "demand"),
    # forecast_models="lightgbm"  # hoặc ("xgboost", "lightgbm"); bỏ dòng để vẽ mọi model
)


In [ ]:
# Compare model performance (numeric table; bar chart is in the figures cell above)
comparison_df = pd.DataFrame(results).T
print("Model Comparison:")
print(comparison_df)

In [ ]:
# Feature importance (bar charts are in the figures cell above)
for model_name in models.keys():
    importance_df = pipeline.model_trainer.get_feature_importance(model_name, top_n=20)
    print(f"\nTop 20 Features - {model_name.upper()}:")
    print(importance_df)